In [1]:
pip install psycopg2


Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install sqlAlchemy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 2.4 MB/s  0:00:01 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [23]:
from sqlalchemy import create_engine, Column, Integer, String, DateTime, Text, MetaData, Table
import pandas as pd

In [24]:
# 读取数据
df = pd.read_csv('chatgpt_reviews_enhanced.csv')
print("Data frame shape:", df.shape)
df.head()


Data frame shape: (20033, 11)


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,c29d08cb-fabe-4a63-80af-acc9b54575db,Jamey Smoot,https://play-lh.googleusercontent.com/a-/ALV-U...,This was a fantastic tool for helping to do re...,1,181,1.2025.266,2025-09-28 11:42:10,NaN,NaN,1.2025.266
1,1c96c79a-a862-4ae9-84f2-c40e105fa55d,Chelsea Megan,https://play-lh.googleusercontent.com/a-/ALV-U...,GPT-5 is horrendous! I am a plus user and even...,1,176,1.2025.266,2025-09-30 06:10:02,NaN,NaN,1.2025.266
2,84409f6c-3357-4c51-b207-8c8891e80db3,A B,https://play-lh.googleusercontent.com/a-/ALV-U...,Updated: Went from 5 stars to 0! Recently chan...,1,47,1.2025.273,2025-10-11 08:47:40,NaN,NaN,1.2025.273
3,fe63f583-4544-41b3-a94a-7fd02feed2c8,Dnyceone,https://play-lh.googleusercontent.com/a/ACg8oc...,My ChatGPT Plus subscription is not syncing to...,1,74,1.2025.266,2025-09-30 20:35:05,NaN,NaN,1.2025.266
4,94c4f86b-1be5-4f5e-bc49-134d7722870d,Michael Breen,https://play-lh.googleusercontent.com/a-/ALV-U...,"in the times I've had the app,it's been great,...",1,75,1.2025.266,2025-09-30 21:35:49,NaN,NaN,1.2025.266


In [32]:
# 创建SQLAlchemy引擎
engine = create_engine('sqlite:///chatgpt_reviews.db')

# 将数据写入数据库
df.to_sql('reviews', engine, if_exists='replace', index=False, 
          dtype={
              'reviewId': String(100),
              'userName': String(200),
              'content': Text,
              'score': Integer,
              'thumbsUpCount': Integer,
              'reviewCreatedVersion': String(50),
              'at': String(50),
              'appVersion': String(50)
          })



20033

In [33]:
# 重新连接验证数据
conn = sqlite3.connect('chatgpt_reviews.db')
cursor = conn.cursor()

# 查看表结构
cursor.execute("PRAGMA table_info(reviews)")
columns = cursor.fetchall()
print("表结构:")
for col in columns:
    print(col)

# 基本统计信息
cursor.execute("""
    SELECT 
        COUNT(*) as total_reviews,
        AVG(score) as avg_score,
        AVG(thumbsUpCount) as avg_thumbs_up,
        MIN(at) as earliest_review,
        MAX(at) as latest_review
    FROM reviews
""")
stats = cursor.fetchone()
print(f"\n数据统计:")
print(f"总评论数: {stats[0]}")
print(f"平均评分: {stats[1]:.2f}")
print(f"平均点赞数: {stats[2]:.2f}")
print(f"最早评论时间: {stats[3]}")
print(f"最新评论时间: {stats[4]}")

conn.close()

表结构:
(0, 'reviewId', 'VARCHAR(100)', 0, None, 0)
(1, 'userName', 'VARCHAR(200)', 0, None, 0)
(2, 'userImage', 'TEXT', 0, None, 0)
(3, 'content', 'TEXT', 0, None, 0)
(4, 'score', 'INTEGER', 0, None, 0)
(5, 'thumbsUpCount', 'INTEGER', 0, None, 0)
(6, 'reviewCreatedVersion', 'VARCHAR(50)', 0, None, 0)
(7, 'at', 'VARCHAR(50)', 0, None, 0)
(8, 'replyContent', 'TEXT', 0, None, 0)
(9, 'repliedAt', 'TEXT', 0, None, 0)
(10, 'appVersion', 'VARCHAR(50)', 0, None, 0)

数据统计:
总评论数: 20033
平均评分: 3.00
平均点赞数: 8.29
最早评论时间: 2023-07-25 08:31:59
最新评论时间: 2025-10-16 13:19:26


In [27]:
# 使用pandas直接连接数据库进行查询
engine = create_engine('sqlite:///chatgpt_reviews.db')

# 从数据库读取数据到DataFrame
db_df = pd.read_sql('SELECT * FROM reviews', engine)
print("从数据库读取的数据形状:", db_df.shape)

# 执行复杂查询
query = """
SELECT 
    appVersion,
    COUNT(*) as review_count,
    AVG(score) as avg_score,
    AVG(thumbsUpCount) as avg_thumbs_up
FROM reviews 
GROUP BY appVersion 
ORDER BY review_count DESC
"""

version_stats = pd.read_sql(query, engine)
print("\n各版本统计:")
print(version_stats)

从数据库读取的数据形状: (20033, 11)

各版本统计:
     appVersion  review_count  avg_score  avg_thumbs_up
0    1.2025.273          4037   3.749071       1.459252
1    1.2025.283          3145   4.583148       0.269952
2    1.2025.266          1319   3.028810       2.576194
3          None          1128   3.218085       1.789007
4    1.2025.203           916   2.024017       5.606987
..          ...           ...        ...            ...
114  1.2024.193             1   3.000000       8.000000
115  1.2024.191             1   2.000000       0.000000
116  1.2023.340             1   3.000000       0.000000
117  1.2023.278             1   3.000000     414.000000
118  1.2023.270             1   2.000000       2.000000

[119 rows x 4 columns]


In [28]:
# 为常用查询字段创建索引
conn = sqlite3.connect('chatgpt_reviews.db')
cursor = conn.cursor()

# 创建索引
indexes = [
    "CREATE INDEX IF NOT EXISTS idx_score ON reviews(score)",
    "CREATE INDEX IF NOT EXISTS idx_appVersion ON reviews(appVersion)",
    "CREATE INDEX IF NOT EXISTS idx_thumbsUp ON reviews(thumbsUpCount)",
    "CREATE INDEX IF NOT EXISTS idx_review_date ON reviews(at)"
]

for index_query in indexes:
    cursor.execute(index_query)

conn.commit()
conn.close()
print("索引创建完成!")

索引创建完成!


In [29]:
# 高级分析查询
def run_advanced_queries():
    engine = create_engine('sqlite:///chatgpt_reviews.db')
    
    # 查询1: 评分分布
    score_dist = pd.read_sql("""
        SELECT score, COUNT(*) as count 
        FROM reviews 
        GROUP BY score 
        ORDER BY score
    """, engine)
    
    # 查询2: 高赞评论
    top_liked = pd.read_sql("""
        SELECT userName, content, score, thumbsUpCount 
        FROM reviews 
        ORDER BY thumbsUpCount DESC 
        LIMIT 10
    """, engine)
    
    # 查询3: 各版本的平均评分趋势
    version_trend = pd.read_sql("""
        SELECT 
            appVersion,
            AVG(score) as avg_score,
            COUNT(*) as review_count
        FROM reviews
        GROUP BY appVersion
        HAVING COUNT(*) > 50  -- 只显示有足够评论的版本
        ORDER BY avg_score DESC
    """, engine)
    
    return score_dist, top_liked, version_trend

score_dist, top_liked, version_trend = run_advanced_queries()

print("评分分布:")
print(score_dist)
print("\n最受欢迎的10条评论:")
print(top_liked)
print("\n各版本评分趋势:")
print(version_trend)

评分分布:
   score  count
0      1   4004
1      2   4009
2      3   4001
3      4   4008
4      5   4011

最受欢迎的10条评论:
              userName                                            content  \
0        Maseena Lewis  I looooove ChatGPT. I really really do. There ...   
1     Austin McPherson  I mostly use chatgpt on PC so it's not usually...   
2           Noa Katzir  HELP concerning error, no option to contact su...   
3         Wayahs Death  I used to love this app. It was a great source...   
4         Shriya yadav  It is an excellent app and I m using it since ...   
5            No Goaway  ChatGPT went from my daily used most favorite ...   
6  Greyson Scarborough  it's better but theres one thing that's bother...   
7         Christina E.  The app itself is unusable now , I often get h...   
8           Ericaacire  I just experienced the biggest whiplash from C...   
9          Lillie Lane  This works really well for a search engine con...   

   score  thumbsUpCount  
0      3   

In [30]:
import shutil
import datetime

# 备份数据库
def backup_database():
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_name = f"chatgpt_reviews_backup_{timestamp}.db"
    shutil.copy2('chatgpt_reviews.db', backup_name)
    print(f"数据库已备份为: {backup_name}")

# 执行备份
backup_database()

数据库已备份为: chatgpt_reviews_backup_20251023_113615.db
